# 02 — The regularized objective: penalising complexity in the loss

*Chapter 09 · XGBoost · Notebook 2 of 5*

In Notebook 1 we found the best leaf weight for any loss, `w* = -G/H`, and reproduced it with XGBoost
once we switched its regularizer **off**. Now we switch it on — and meet the idea that gives XGBoost
its name.

Until now, complexity was held in check from the *outside*: a maximum depth, a number of trees
(Chapters 04 and 08). XGBoost instead writes the cost of complexity directly **into the objective it
minimises**, through two penalties — `λ` on the size of each leaf weight, and `γ` on the number of
leaves. We will add those penalties by hand, watch the leaf weight shrink, derive the rule XGBoost uses
to decide whether a split is worth making, and check it against the library.

**Prerequisites.** Notebook 1 (the second-order leaf weight `w* = -G/H`, with `G = Σ g`, `H = Σ h`);
Chapter 04 (a split is chosen by the gain it brings); Chapter 08 (`init_`, the constant starting model).

**What you'll be able to do.**
- Write XGBoost's regularized objective `Ω(f) = γ·T + ½λ·Σ w²` and say what each penalty does.
- Show the regularized leaf weight is `w* = -G/(H + λ)`, and that `λ` shrinks it toward zero.
- Derive the **split gain** from the structure score, and read `γ` as the price of a split.
- Match XGBoost's leaf weights exactly, and explain why its reported gain is twice the textbook's.

## Where Notebook 1 left us

The optimal weight for a leaf is `w* = -G/H`: the gradient sum `G` over the curvature sum `H`. XGBoost,
with `reg_lambda = 0` and `gamma = 0`, put exactly that value in its leaves. Those two knobs were the
regularizer, switched off. Turn them on and XGBoost charges for complexity **inside** the objective —
not only through `max_depth` or the number of trees, as in Chapters 04 and 08. That is the structural
idea this notebook builds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from xgboost import XGBRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()

SEED = 0

# A small 1-D regression toy, legible enough to read XGBoost's one tree by eye.
x = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
y = np.array([1.0, 1.2, 0.9, 5.0, 5.3, 4.8])
F0 = float(y.mean())     # the constant starting model: the mean (the natural F0 for squared error)
g = F0 - y               # gradient of 1/2 (y - F)^2 at F0:  g = F - y
h = np.ones_like(y)      # squared error has constant curvature h = 1

# The candidate split x < 2.5 separates the three cheap points from the three dear ones.
left = x.ravel() < 2.5
GL, HL = g[left].sum(), h[left].sum()
GR, HR = g[~left].sum(), h[~left].sum()
G, H = g.sum(), h.sum()

print(f"F0 = mean(y) = {F0:.4f}")
print(f"left  (x<2.5):  G_L = {GL:+.3f}   H_L = {HL:.0f}")
print(f"right (x>2.5):  G_R = {GR:+.3f}   H_R = {HR:.0f}")
print(f"root:           G   = {G:+.3f}   H   = {H:.0f}")


## The regularized objective

XGBoost minimises the loss **plus** a complexity penalty on the tree `f`:

`Obj = Σ loss + Ω(f)`,    with    `Ω(f) = γ·T + ½λ·Σⱼ wⱼ²`

where `T` is the number of leaves and `wⱼ` their weights. Two knobs:

- **`λ` (`reg_lambda`)** charges for *large* leaf weights — an L2 penalty, the same shape as ridge
  regression or the inverse of logistic regression's `C` (Chapter 03).
- **`γ` (`gamma`)** charges a flat cost for *each* leaf.

The model now prefers a smaller tree — fewer leaves, smaller weights — unless the data genuinely earns
the extra complexity. Let us see each penalty act.

## Effect 1 — λ shrinks the leaf

Add the `½λw²` penalty to a single leaf's second-order objective from Notebook 1. The leaf holds the
points routed to it; its objective in the weight `w` is

`G·w + ½(H + λ)·w²`

— the same parabola as before, but with `λ` added to the curvature. Minimising it (set the derivative
to zero) gives

`w* = -G / (H + λ)`

The `λ` sits in the denominator, so every leaf is pulled **toward zero**. It is Notebook 1's leaf
weight, now reined in.

In [ ]:
# The right leaf's objective  G_R*w + 1/2 (H_R + lambda) w^2,  for three values of lambda.
ws = np.linspace(-0.6, 3.0, 400)

fig, ax = plt.subplots(figsize=(7.5, 5))
for lam, col in zip([0.0, 1.0, 10.0], ["model", "class_b", "error"], strict=True):
    obj = GR * ws + 0.5 * (HR + lam) * ws**2
    w_star = -GR / (HR + lam)
    ax.plot(ws, obj, color=COLORS[col], linewidth=2.4, label=f"λ={lam:.0f} → w*={w_star:.3f}")
    ax.plot(w_star, GR * w_star + 0.5 * (HR + lam) * w_star**2, "o", color=COLORS[col],
            markersize=11, zorder=5)
ax.axvline(0.0, color=COLORS["zero"], linewidth=1.0)
ax.set_xlabel("leaf weight  w")
ax.set_ylabel("leaf objective  (lower = better)")
ax.legend()
plt.show()

print("right-leaf  w* = -G_R / (H_R + λ):")
for lam in [0, 1, 3, 10, 100]:
    print(f"  λ = {lam:4d}   ->   w* = {-GR / (HR + lam):+.4f}")


**Read the figure.** Each parabola is the same leaf scored with a different `λ`. As `λ` grows the
bowl steepens and its lowest point slides toward zero: `+2.0` at `λ=0`, `+1.5` at `λ=1`, `+0.46` at
`λ=10`. The leaf still points the right way — it is still positive — but takes a shorter, safer step.
At `λ=0` we are back at Notebook 1's unregularized leaf; crank `λ` up and the tree turns timid. `λ` is
a dial between the two.

## Effect 2 — scoring a whole tree

To decide *where* (and *whether*) to split, XGBoost needs to score a tree's shape. Substitute the best
leaf weight `w* = -G/(H+λ)` back into a leaf's objective:
`G·w* + ½(H+λ)·w*² = -G²/(H+λ) + ½G²/(H+λ) = -½ G²/(H+λ)` — the ½ is born in this cancellation. Summing
over the leaves, the whole tree scores

`Obj(structure) = -½ Σ_leaves G²/(H + λ)  +  γT`        (Chen & Guestrin, eq. 6)

Lower is better. A leaf earns its negative (good) contribution by having a large gradient sum `|G|`
relative to its curvature `H + λ`. The `γT` term adds back a flat charge for every leaf we create.

## The gain of a split (and a sign to watch)

A split takes one leaf and replaces it with two children. Its **gain** is how much the structure score
improves — `score(before) − score(after)`:

`gain = ½[ G_L²/(H_L+λ) + G_R²/(H_R+λ) − G²/(H+λ) ] − γ`        (eq. 7)

The bracket is "children minus parent". A **positive** gain means the (negative) objective got *more*
negative — the split helped — so XGBoost keeps the split with the largest positive gain.

A sign to watch: the structure score is *negative*, so reasoning loosely as "parent loss minus children
loss" flips the sign. We derived the gain from the score (eq. 6) precisely to keep it straight — the
formula is `score(before) − score(after)`, not the other way round.

In [ ]:
lam = 1.0
term_L = GL**2 / (HL + lam)
term_R = GR**2 / (HR + lam)
term_parent = G**2 / (H + lam)
gain_half = 0.5 * (term_L + term_R - term_parent)   # textbook eq. 7, with gamma = 0

print(f"with λ = {lam:.0f},  γ = 0:")
print(f"  G_L²/(H_L+λ) = {term_L:.4f}")
print(f"  G_R²/(H_R+λ) = {term_R:.4f}")
print(f"  G²/(H+λ)     = {term_parent:.4f}   (the parent leaf)")
print(f"  ½-gain = ½·[{term_L:.1f} + {term_R:.1f} − {term_parent:.1f}] = {gain_half:.4f}")


## γ — the price of a split

The `−γ` at the end of the gain is a toll: a split is kept only if its gain clears `γ`. So `γ` acts as
**pre-pruning** — it refuses a split that does not pay for itself. (On deeper trees `γ` also prunes
already-grown branches; here, with a single split, it blocks that split outright.) Raise `γ` and
shallower trees result. Let us watch the toll decide, by measuring XGBoost directly rather than
assuming where the threshold falls.

In [ ]:
# XGBoost reports the gain WITHOUT the 1/2; that no-1/2 value is also the unit of gamma.
xgb_gain = 2.0 * gain_half        # we confirm this against the library in the next cell

# Measure, do not assume: fit XGBoost at each gamma and see whether the split survives.
gammas = [0, 5, 10, 15, 18, 19, 22, 25]
kept = []
for gam in gammas:
    r = XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=1.0,
                     gamma=float(gam), base_score=F0, objective="reg:squarederror",
                     tree_method="exact", random_state=SEED).fit(x, y)
    n_internal = int((r.get_booster().trees_to_dataframe()["Feature"] != "Leaf").sum())
    kept.append(n_internal == 1)

net = [xgb_gain - gam for gam in gammas]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.axhline(0.0, color=COLORS["zero"], linewidth=1.2)
ax.plot(gammas, net, color=COLORS["muted"], linewidth=1.6, zorder=1)
for gam, n, k in zip(gammas, net, kept, strict=True):
    ax.plot(gam, n, "o", color=COLORS["correct"] if k else COLORS["error"], markersize=13, zorder=5)
ax.plot([], [], "o", color=COLORS["correct"], label="split kept")
ax.plot([], [], "o", color=COLORS["error"], label="split pruned")
ax.set_xlabel("γ  (min_split_loss)")
ax.set_ylabel("gain − γ   (XGBoost units; gain = 2×9 = 18)")
ax.legend()
plt.show()

print("γ sweep measured on XGBoost (λ = 1):")
for gam, k in zip(gammas, kept, strict=True):
    print(f"  γ = {gam:3d}   ->   {'SPLIT kept' if k else 'PRUNED (single leaf)'}")


**Read the figure — and an honest detail.** The split survives while `gain − γ` stays at or above
zero, and is pruned the moment `γ` passes the gain. The crossover is the key: our textbook ½-gain was
**9**, but XGBoost keeps the split up to `γ = 18` and prunes at `γ = 19`.

XGBoost works in **no-½ units, consistently**. Both the gain it reports *and* the `γ` it compares
against are the no-½ value, `18`. The ½ in the textbook is a unit convention — it scales the gain and
the `γ` threshold by the same factor, so the winning split and the prune decision are identical either
way. The practical consequence is worth remembering: **`γ` lives in the same units XGBoost prints as
`Gain`** — to suppress a split whose reported `Gain` is 18, set `γ` above 18, not above 9.

In [ ]:
# Fit the one tree with the regularizer ON (lambda = 1), and compare to our hand calculation.
reg = XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=1.0, gamma=0.0,
                   base_score=F0, objective="reg:squarederror", tree_method="exact",
                   random_state=SEED).fit(x, y)
dump = reg.get_booster().trees_to_dataframe()
xgb_leaves = sorted(float(v) for v in dump.loc[dump["Feature"] == "Leaf", "Gain"])
xgb_gain_reported = float(dump.loc[dump["Feature"] != "Leaf", "Gain"].iloc[0])
byhand_leaves = sorted([-GL / (HL + 1.0), -GR / (HR + 1.0)])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))
idx = np.arange(2)
axL.bar(idx - 0.18, byhand_leaves, width=0.34, color=COLORS["model"], label="by hand  -G/(H+λ)")
axL.bar(idx + 0.18, xgb_leaves, width=0.34, color=COLORS["highlight"], label="XGBoost")
axL.axhline(0.0, color=COLORS["zero"], linewidth=1.0)
axL.set_xticks(idx)
axL.set_xticklabels(["left leaf", "right leaf"])
axL.set_ylabel("leaf weight  w*")
axL.set_title("leaf weights: identical")
axL.legend()

axR.bar([0, 1], [gain_half, xgb_gain_reported], width=0.55,
        color=[COLORS["model"], COLORS["highlight"]])
axR.set_xticks([0, 1])
axR.set_xticklabels(["by hand  ½[…]", "XGBoost  reported"])
axR.set_ylabel("split gain")
axR.set_title("gain: XGBoost reports 2× (the ½ it drops)")
plt.show()

print(dump[["ID", "Feature", "Split", "Gain", "Cover"]].to_string(index=False))
print()
byhand_round = [round(float(v), 4) for v in byhand_leaves]
xgb_round = [round(float(v), 4) for v in xgb_leaves]
print(f"by-hand leaves {byhand_round}  ==  XGBoost {xgb_round}  (to float precision)")
print(f"½-gain {gain_half:.1f}   ·   XGBoost reported Gain {xgb_gain_reported:.1f}   "
      f"(ratio {xgb_gain_reported / gain_half:.1f})")


**Read the figure.** The leaf weights match exactly — by hand and from XGBoost, both `±1.5` at
`λ=1`. The gains differ only by the ½ explained above. And note the `Cover` column in the printed
tree: it is **`Σ h`**, the sum of the curvatures in each node (here `3` per leaf, which equals the
count, because squared error has `h=1`). `Cover` is the quantity the parameter `min_child_weight` will
threshold in Notebook 4 — a floor on how much curvature a leaf must hold.

One note on `base_score`: we pinned it to `mean(y)` so that `F₀` was a known constant for the
hand-check. Left free, XGBoost **learns** it from the data — for this toy, the mean — which is the same
role Chapter 08's `init_` played.

## Where this sits

`λ` and `γ` move complexity control **into the objective**. Chapters 04 and 08 limited a tree only from
the outside — its depth, the number of trees. XGBoost prices complexity inside the loss it optimises,
so every split must justify itself against `γ`, and every leaf weight is shrunk by `λ`. The engine is
unchanged — it is still Notebook 1's second-order leaf — but it is now regularized.

The next notebook turns to a different, very practical strength: how XGBoost splits a feature when some
of its values are **missing**.

## Your turn

1. **(warm-up)** For the right leaf (`G = -6`, `H = 3`), compute `w* = -G/(H+λ)` at `λ = 0, 1, 10` and
   confirm the shrinkage you see in Figure 1. What does `λ → ∞` drive every leaf toward?
2. **(core)** Compute the ½-gain of the `x < 2.5` split by hand from `G_L, H_L, G_R, H_R` (with
   `λ = 1, γ = 0`), confirm XGBoost reports exactly twice your number, and find the smallest integer `γ`
   that prunes the split.
3. **(reach)** Show that raising `λ` lowers *every* split's gain (look at how `λ` enters each
   `G²/(H+λ)` term), so `λ` discourages splitting on top of shrinking leaves. How is that different from
   what `γ` does? (One penalises the *size* of a leaf's answer; the other charges a flat fee per leaf.)

## What you built

You wrote XGBoost's regularized objective `Ω = γ·T + ½λ·Σ w²` and watched both penalties act:

- `λ` shrinks each leaf to `w* = -G/(H + λ)` — Notebook 1's leaf, pulled toward zero.
- the **split gain** `½[ G_L²/(H_L+λ) + G_R²/(H_R+λ) − G²/(H+λ) ] − γ`, derived from the structure
  score, decides which split (if any) to make; `γ` is the price each split must clear.

XGBoost reproduced the leaf weights exactly and reported the gain in no-½ units (twice the textbook),
pruning once `γ` exceeded it — the same decisions, a unit convention apart.

**Vocabulary.** regularized objective · L2 leaf penalty `λ` (`reg_lambda`) · per-leaf cost `γ`
(`gamma`) · structure score · split gain · `Cover = Σ h`.

Next: **sparsity-aware splits** — how XGBoost sends missing values down a learned default direction,
with no imputation at all.

## References

- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (Eq. 6: the structure score and
  optimal leaf weight; Eq. 7: the split gain with the `λ` and `γ` penalties.)
- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.10. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

*Previous: 01 — The second-order view.*  ·  *Next: 03 — Sparsity-aware splits.*